In [0]:
--Incremental load into fact_stock_daily
merge into analytics.fact_stock_daily target
using (
  select *
  from analytics.stock_prices
) source
on target.ticker = source.ticker
and target.trade_date = source.trade_date

when not matched then insert *;

--Incremental load into stock_daily_metrics

create or replace temp view base as

with max_date as (
  select coalesce(max(trade_date), '1900-01-01') as max_trade_date
  from analytics.stock_daily_metrics
)

select 
  f.ticker,
  f.trade_date,
  f.close, 

  lag(f.close) over (
    partition by f.ticker
    order by f.trade_date
  ) as prev_close,

  avg(f.close) over (
    partition by f.ticker
    order by f.trade_date
    rows between 9 preceding and current row
  ) as ma_10,

  avg(f.close) over (
    partition by f.ticker
    order by f.trade_date
    rows between 19 preceding and current row
  ) as ma_20,

  avg(f.close) over (
    partition by f.ticker
    order by f.trade_date
    rows between 49 preceding and current row
  ) as ma_50

from analytics.fact_stock_daily f
cross join max_date m
where f.trade_date >= date_sub(m.max_trade_date, 50);

merge into analytics.stock_daily_metrics target
using (
  select
    ticker,
    trade_date,
    close,
    (close - prev_close) / nullif(prev_close, 0) as daily_return,
    ma_10,
    ma_20,
    ma_50
  from base
) source

on target.ticker = source.ticker
and target.trade_date = source.trade_date

when matched then update set
  target.close = source.close,
  target.daily_return = source.daily_return,
  target.ma_10 = source.ma_10,
  target.ma_20 = source.ma_20,
  target.ma_50 = source.ma_50

when not matched then insert (
  ticker,
  trade_date,
  close, 
  daily_return,
  ma_10,
  ma_20,
  ma_50
)
values (
  source.ticker,
  source.trade_date,
  source.close,
  source.daily_return,
  source.ma_10,
  source.ma_20,
  source.ma_50
);

drop view if exists base;


-- Incremental load for mart_stock_features

merge into analytics.mart_stock_features t
using (

  with base as (

    select
      ticker,
      trade_date,
      close,
      volume,

      lag(close) over (
        partition by ticker
        order by trade_date
      ) as prev_close
    from analytics.fact_stock_daily

    where trade_date >= coalesce(
      (
      select dateadd(day, -120, max(trade_date))
        from analytics.mart_stock_features
    ),
    '1900-01-01'
    )
  ),

  returns as (
      select *,
      (close - prev_close) / nullif(prev_close, 0) as daily_return
    from base 
    where prev_close is not null
  ),

  rolling as (
    select *,
      (close /lag(close, 5) over (partition by ticker order by trade_date)) - 1 as return_5d,
      (close /lag(close, 20) over (partition by ticker order by trade_date)) - 1 as return_20d,
      (close /lag(close, 50) over (partition by ticker order by trade_date)) - 1 as return_50d,

      stddev(daily_return) over (
        partition by ticker
        order by trade_date
        rows between 20 preceding and 1 preceding
      ) as vol_20d,

      avg(volume) over (
        partition by ticker
        order by trade_date
        rows between 20 preceding and 1 preceding
      ) as avg_volume_20d,

      max(close) over (
        partition by ticker
        order by trade_date
        rows between 20 preceding and 1 preceding
      ) as high_20d

    from returns
  ),

  features as (
    select *,
      (close / high_20d) as pct_of_20d_high,

      case 
        when volume > avg_volume_20d then 1 else 0
      end as is_volume_spike,

      case 
        when close >= high_20d * 0.99 
          and return_20d > 0
          and volume > avg_volume_20d
        then 1 else 0
      end as is_breakout_candidate,

      row_number() over (
        partition by trade_date
        order by return_20d desc
      ) as momentum_rank

    from rolling
  )

    select *
    from features
    where return_50d is not null
    and vol_20d is not null
    and avg_volume_20d is not null
) s

  on t.ticker = s.ticker
  and t.trade_date = s.trade_date

  when matched then update set
    close = s.close,
    volume = s.volume,
    daily_return = s.daily_return,
    return_5d = s.return_5d,
    return_20d = s.return_20d,
    return_50d = s.return_50d,
    vol_20d = s.vol_20d,
    avg_volume_20d = s.avg_volume_20d,
    high_20d = s.high_20d,
    pct_of_20d_high = s.pct_of_20d_high,
    is_volume_spike = s.is_volume_spike,
    is_breakout_candidate = s.is_breakout_candidate,
    momentum_rank = s.momentum_rank
  
  when not matched then insert (
    ticker,
    trade_date,
    close,
    volume,
    prev_close,
    daily_return,
    return_5d,
    return_20d,
    return_50d,
    vol_20d,
    avg_volume_20d,
    high_20d,
    pct_of_20d_high,
    is_volume_spike,
    is_breakout_candidate,
    momentum_rank
  )
  values (
    s.ticker,
    s.trade_date,
    s.close,
    s.volume,
    s.prev_close,
    s.daily_return,
    s.return_5d,
    s.return_20d,
    s.return_50d,
    s.vol_20d,
    s.avg_volume_20d,
    s.high_20d,
    s.pct_of_20d_high,
    s.is_volume_spike,
    s.is_breakout_candidate,
    s.momentum_rank
  );


